In [22]:
import os
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import pdfplumber
import re
from fpdf import FPDF

In [20]:
!pip install fpdf

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40770 sha256=d04160c4e92034461b9d92a205e7489413321cbf28caef726ee16df2632141c1
  Stored in directory: c:\users\admin\appdata\local\pip\cache\wheels\f9\95\ba\f418094659025eb9611f17cbcaf2334236bf39a0c3453ea455
Successfully built fpdf


In [27]:
# ==========================================
# 1. TẢI CẤU HÌNH KẾT NỐI TỪ FILE .ENV
# ==========================================
load_dotenv(override=True)
host = os.getenv('MYSQL_SERVER')
database = os.getenv('MYSQL_DB')
username = os.getenv('MYSQL_USER')
password = os.getenv('MYSQL_PASS')

def get_db_connection():
    """Hàm hỗ trợ mở kết nối nhanh đến MySQL"""
    try:
        return mysql.connector.connect(
            host=host, database=database, user=username, password=password
        )
    except Exception as e:
        print(f" Lỗi kết nối Database: {e}")
        return None

In [28]:
# ==========================================
# 2. BỘ PHẬN AI ĐỌC VÀ TRÍCH XUẤT HÓA ĐƠN
# ==========================================
def parse_invoice(file_path):
    """
    Khu vực này dùng để viết code AI (như Tesseract OCR, pdfplumber, 
    hoặc API của OpenAI/Gemini) để đọc chữ trên hóa đơn.
    """
    print(f"⏳ Đang dùng AI phân tích hóa đơn: {file_path}...")
    
    # Dữ liệu giả lập (AI sau khi đọc xong sẽ trả về một bộ từ điển thế này)
    # Bạn sẽ thay phần này bằng code trích xuất thật sau.
    extracted_data = {
        'invoice_id': 'INV-2026-001',
        'vendor_name': 'Cong ty ABC',
        'total_amount': 1500000.0,
        'invoice_date': '2026-07-17'
    }
    return extracted_data

In [30]:


# 1. Xác định đường dẫn thư mục data (lùi 1 cấp từ scripts_python)
thumuc_data = "../data"

# Kiểm tra nếu chưa có thư mục data thì tự tạo
if not os.path.exists(thumuc_data):
    os.makedirs(thumuc_data)

# Đường dẫn file đích
duong_dan_file = os.path.join(thumuc_data, "hoa_don_demo.pdf")

# 2. Dùng fpdf để vẽ hóa đơn
pdf = FPDF()
pdf.add_page()

# Cài đặt font chữ
pdf.set_font("Arial", 'B', 16)
pdf.cell(200, 10, txt="INVOICE DEMO", ln=True, align='C')
pdf.ln(10)

# Điền nội dung (cố tình viết chuẩn form để Regex đọc được)
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, txt="Vendor Name: Cong ty TNHH Cloud Services", ln=True)
pdf.cell(200, 10, txt="Invoice No: INV-2026-07-001", ln=True)
pdf.cell(200, 10, txt="Date: 2026-07-17", ln=True)
pdf.ln(10)
pdf.set_font("Arial", 'B', 14)
pdf.cell(200, 10, txt="Total: 1,500,000.00", ln=True)

# 3. Xuất file
pdf.output(duong_dan_file)

print(f"Đã tạo thành công file PDF tại: {duong_dan_file}")

Đã tạo thành công file PDF tại: ../data\hoa_don_demo.pdf


In [47]:
# ==========================================
# 3. NẠP DỮ LIỆU VÀO DATABASE FINOPS
# ==========================================
def save_to_mysql(data):
    """Lưu dữ liệu AI vừa đọc được vào bảng trong MySQL"""
    conn = get_db_connection()
    if conn is None: 
        return
        
    cursor = conn.cursor()
    
    # LƯU Ý QUAN TRỌNG: Hãy sửa 'ten_bang_hoa_don' thành tên bảng thực tế của bạn
    sql_query = """
        INSERT INTO saas_invoices (invoice_id, vendor_name, total_amount, invoice_date)
        VALUES (%s, %s, %s, %s)
    """
    # Khớp dữ liệu vào câu lệnh SQL
    values = (
    data['invoice_id'],     # Sẽ được nạp vào cột 'invoice_number'
    data['vendor_name'],    # Sẽ được nạp vào cột 'vendor_id'
    data['total_amount'],   # Sẽ được nạp vào cột 'amount'
    data['invoice_date']    # Sẽ được nạp vào cột 'invoice_date'
)
    
    try:
        cursor.execute(sql_query, values)
        conn.commit() # Xác nhận lưu
        print(f" Đã lưu dữ liệu hóa đơn vào MySQL thành công!")
    except Exception as e:
        print(f" Có lỗi khi lưu dữ liệu: {e}")
    finally:
        cursor.close()
        conn.close()

In [54]:
def save_to_mysql(data):
    """Lưu dữ liệu AI vừa đọc được vào bảng trong MySQL"""
    conn = get_db_connection()
    if conn is None:
        return

    cursor = conn.cursor()

    # 1. Cập nhật câu lệnh SQL với đúng 3 cột đang có trong Database
    sql_query = """
        INSERT INTO saas_invoices (InvoiceID, InvoiceDate, TotalAmount)
        VALUES (%s, %s, %s)
    """
    
    # 2. Truyền 3 giá trị tương ứng từ AI vào đúng thứ tự
    values = (
        data['invoice_id'], 
        data['invoice_date'], 
        data['total_amount']
    )

    try:
        cursor.execute(sql_query, values)
        conn.commit() # Bấm nút "Save" xuống ổ cứng
        print(f" Đã lưu dữ liệu hóa đơn vào MySQL thành công!")
    except Exception as e:
        print(f" Có lỗi khi lưu dữ liệu: {e}")
    finally:
        cursor.close()
        conn.close()

In [55]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 1. Kết nối vào Database
conn = mysql.connector.connect(
    host=os.getenv('MYSQL_SERVER'),
    database=os.getenv('MYSQL_DB'),
    user=os.getenv('MYSQL_USER'),
    password=os.getenv('MYSQL_PASS')
)

cursor = conn.cursor()

# 2. Chạy lệnh yêu cầu MySQL liệt kê tất cả các bảng
cursor.execute("SHOW TABLES;")

# 3. Lấy kết quả và in ra màn hình
tables = cursor.fetchall()

print(f" Danh sách các bảng đang có trong Database '{os.getenv('MYSQL_DB')}':")

if not tables:
    print("  -> (Database của bạn hiện chưa có bảng nào!)")
else:
    for table in tables:
        # Lấy phần tử đầu tiên của tuple trả về
        print(f"  - {table[0]}")

# 4. Đóng kết nối
cursor.close()
conn.close()

 Danh sách các bảng đang có trong Database 'FinOps_ShadowIT':
  - hr_employees
  - saas_catalog
  - saas_invoices
  - sso_login_log
  - vw_user_software_activity


In [56]:
file_test = "../data/hoa_don_demo.pdf"


# 1. Chạy bóc tách
du_lieu = parse_invoice(file_test)
print(f" Dữ liệu thật bóc được: {du_lieu}")

# 2. Lưu Database 
save_to_mysql(du_lieu)


 Đang đọc và bóc tách chữ từ file: ../data/hoa_don_demo.pdf...
 Đã quét xong chữ! Đang phân tích thông tin...
 Dữ liệu thật bóc được: {'invoice_id': 'INV-2026-07-001', 'vendor_name': 'Cong ty TNHH Cloud Services', 'total_amount': 1500000.0, 'invoice_date': '2026-07-17'}
 Có lỗi khi lưu dữ liệu: 1062 (23000): Duplicate entry 'INV-2026-07-001' for key 'saas_invoices.PRIMARY'


In [43]:
# Mở kết nối (Sử dụng lại hàm đã viết sẵn của bạn)
conn = get_db_connection()
cursor = conn.cursor()

# Lệnh DESCRIBE giúp xem cấu trúc các cột trong một bảng
cursor.execute("DESCRIBE saas_invoices;")
columns = cursor.fetchall()

print("Mô tả cấu trúc bảng 'saas_invoices':")
print(f"{'Tên cột':<20} | {'Kiểu dữ liệu'}")
print("-" * 40)

for col in columns:
    print(f"{col[0]:<20} | {col[1]}")

cursor.close()
conn.close()


Mô tả cấu trúc bảng 'saas_invoices':
Tên cột              | Kiểu dữ liệu
----------------------------------------
InvoiceID            | varchar(50)
InvoiceDate          | date
AppID                | int
TotalLicenses        | int
TotalAmount          | decimal(12,2)
PaidByDepartment     | varchar(50)


In [57]:
# ==========================================
# 2. BỘ PHẬN AI ĐỌC VÀ TRÍCH XUẤT HÓA ĐƠN THẬT
# ==========================================
def parse_invoice(file_path):
    print(f" Đang đọc và bóc tách chữ từ file: {file_path}...")
    
    extracted_data = {
        'invoice_id': 'Unknown',
        'vendor_name': 'Unknown',
        'total_amount': 0.0,
        'invoice_date': '1970-01-01'
    }
    
    try:
        with pdfplumber.open(file_path) as pdf:
            text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                    
        print(f" Đã quét xong chữ! Đang phân tích thông tin...")
        
        # Tìm Tên nhà cung cấp
        vendor_match = re.search(r'Vendor Name:\s*(.+)', text, re.IGNORECASE)
        if vendor_match: extracted_data['vendor_name'] = vendor_match.group(1).strip()
            
        # Tìm Số hóa đơn
        inv_match = re.search(r'Invoice No:\s*([A-Z0-9\-]+)', text, re.IGNORECASE)
        if inv_match: extracted_data['invoice_id'] = inv_match.group(1).strip()
            
        # Tìm Ngày tháng
        date_match = re.search(r'Date:\s*([\d]{4}-[\d]{2}-[\d]{2})', text, re.IGNORECASE)
        if date_match: extracted_data['invoice_date'] = date_match.group(1).strip()
            
        # Tìm Tổng tiền
        money_match = re.search(r'Total:\s*([\d\,\.]+)', text, re.IGNORECASE)
        if money_match:
            tien_str = money_match.group(1).replace(',', '')
            extracted_data['total_amount'] = float(tien_str)
            
    except Exception as e:
        print(f" Có lỗi khi đọc file PDF: {e}")
        
    return extracted_data

In [58]:
# 1. Mở kết nối đến Database
conn = get_db_connection()

if conn is not None:
    # 2. Viết câu lệnh SELECT 
    # (Lấy toàn bộ dữ liệu từ bảng saas_invoices)
    sql_query = "SELECT * FROM saas_invoices;"
    
    print("Đang truy vấn dữ liệu từ Database...\n")
    
    try:
        # 3. Dùng Pandas hứng dữ liệu và chuyển thành DataFrame
        df_invoices = pd.read_sql(sql_query, conn)
        
        # Kiểm tra xem bảng có trống không
        if df_invoices.empty:
            print(f" Bảng saas_invoices hiện chưa có dữ liệu nào.")
        else:
            print(f" Dữ liệu hiện có trong bảng saas_invoices:")
            # Dùng hàm display() của Jupyter để in bảng với định dạng đẹp nhất
            display(df_invoices)
            
    except Exception as e:
        print(f" Có lỗi khi truy vấn: {e}")
        
    finally:
        # 4. Luôn nhớ đóng kết nối
        conn.close()

Đang truy vấn dữ liệu từ Database...

 Dữ liệu hiện có trong bảng saas_invoices:


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13020\2217165744.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_invoices = pd.read_sql(sql_query, conn)


,InvoiceID,InvoiceDate,AppID,TotalLicenses,TotalAmount,PaidByDepartment
0,INV-2026-07-001,2026-07-17,NaN,NaN,1500000.0,None
1,INV-202604-CANVA,2026-04-12,4.0,30.0,360.0,Marketing
2,INV-202604-M365,2026-04-05,1.0,150.0,3000.0,IT
3,INV-202604-ZOOM,2026-04-10,2.0,100.0,1500.0,IT
4,Unknown,1970-01-01,NaN,NaN,0.0,None


In [59]:
# 1. Mở kết nối
conn = get_db_connection()
cursor = conn.cursor()

# 2. Viết lệnh DELETE để xóa các hóa đơn bị lỗi (Unknown)
sql_delete = "DELETE FROM saas_invoices WHERE InvoiceID = 'Unknown';"

try:
    cursor.execute(sql_delete)
    conn.commit()
    so_dong_da_xoa = cursor.rowcount
    print(f" Đã dọn dẹp thành công! Xóa {so_dong_da_xoa} dòng dữ liệu rác khỏi Database.")
except Exception as e:
    print(f" Có lỗi khi xóa dữ liệu: {e}")
finally:
    cursor.close()
    conn.close()

 Đã dọn dẹp thành công! Xóa 1 dòng dữ liệu rác khỏi Database.
